# Suite VLGPRT — Printing Verilog

The printer writes one fixed layout and the parentheses precedence needs; printing validates the tree and checks
the standard first.

In [ ]:
from mbse.Programs.Framework import Errors
from mbse.Programs.Verilog import SystemVerilog2023, Syntax as S, Verilog2005
from support import raises

P = SystemVerilog2023.print


def name(spelling):
    return S.Identifier(spelling=spelling)


def ref(spelling):
    return S.NameExpression(name=name(spelling))


def number(spelling="1"):
    return S.IntegerLiteral(spelling=spelling)


def binary(left, operator, right):
    return S.BinaryExpression(left=left, operator=operator, right=right)


def roundtrip(text):
    """Text reads and prints back to itself."""
    printed = P(SystemVerilog2023.parse(text))
    assert printed == text, printed
    return printed

## VLGPRT-01 · Parentheses follow IEEE 1800's precedence, and right-associative operators

In [ ]:
a, b, c = ref("a"), ref("b"), ref("c")
assert P(binary(binary(a, "+", b), "*", c)) == "(a + b) * c"
assert P(binary(a, "-", binary(b, "-", c))) == "a - (b - c)" and P(binary(binary(a, "-", b), "-", c)) == "a - b - c"
assert P(binary(a, "->", binary(b, "->", c))) == "a -> b -> c" and P(binary(binary(a, "->", b), "->", c)) == "(a -> b) -> c"
def cond():
    return S.ConditionalExpression(condition=ref("a"), consequence=ref("b"), alternative=ref("c"))


assert P(S.ConditionalExpression(condition=cond(), consequence=ref("b"), alternative=cond())) == "(a ? b : c) ? b : a ? b : c"
assert P(S.UnaryExpression(operator="-", operand=binary(a, "+", b))) == "-(a + b)"
assert P(S.UnaryExpression(operator="~", operand=S.UnaryExpression(operator="-", operand=a))) == "~(-a)"
assert P(S.IndexExpression(value=binary(a, "+", b), index=number())) == "(a + b)[1]"
assert P(S.InsideExpression(value=binary(a, "==", b), set=[number()])) == "(a == b) inside {1}"
assert P(binary(a, "+", S.InsideExpression(value=b, set=[c]))) == "a + (b inside {c})"
assert P(binary(S.AssignmentExpression(target=a, operator="=", value=b), "+", c)) == "(a = b) + c"
assert P(S.IncrementExpression(operator="++", operand=binary(a, "+", b))) == "++(a + b)"
assert P(S.CastExpression(type=binary(a, "+", b), value=c)) == "(a + b)'(c)"
def clocked(sequence):
    return S.ClockedSequence(clock=S.EventControl(events=[S.EventExpression(expression=ref("k"))]), sequence=sequence)


def always(operand):
    return S.UnaryProperty(operator="always", operand=operand)


def step(delay, sequence):
    return S.DelayStep(delay=delay, sequence=sequence)


assert P(S.BinaryProperty(left=always(a), operator="and", right=b)) == "(always a) and b"
assert P(S.BinaryProperty(left=S.BinaryProperty(left=a, operator="and", right=always(b)), operator="or", right=c)) ==     "a and (always b) or c"
assert P(S.BinaryProperty(left=a, operator="and", right=always(b))) == "a and always b"  # nothing follows it
assert P(S.DelaySequence(first=S.BinarySequence(left=a, operator="or", right=b), steps=[step(number(), c)])) == "(a or b) ##1 c"
assert P(S.RepetitionSequence(sequence=S.DelaySequence(first=a, steps=[step(number(), b)]), operator="*",
                              count=number("2"))) == "(a ##1 b)[*2]"
assert P(S.ConditionalProperty(condition=a, consequence=always(b), alternative=c)) == "if (a) (always b) else c"
throughout = S.BinarySequence(left=b, operator="throughout", right=c)
assert P(S.BinarySequence(left=a, operator="throughout", right=throughout)) == "a throughout b throughout c"
assert P(S.BinarySequence(left=S.BinarySequence(left=a, operator="throughout", right=b), operator="throughout", right=c)) ==     "(a throughout b) throughout c"
assert P(S.UnaryProperty(operator="not", operand=S.BinaryProperty(left=a, operator="and", right=b))) == "not (a and b)"
assert P(S.DelaySequence(steps=[step(binary(a, "+", number()), b), step(number("2"), clocked(c))])) == "##(a + 1) b ##2 @(k) c"
assert P(S.DelaySequence(steps=[step(number("2"), clocked(c)), step(number(), ref("d"))])) == "##2 (@(k) c) ##1 d"
assert P(S.DelaySequence(first=clocked(a), steps=[step(number(), b)])) == "(@(k) a) ##1 b"
assert P(S.ImplicationProperty(antecedent=a, operator="|->", consequent=S.ImplicationProperty(
    antecedent=b, operator="|=>", consequent=c))) == "a |-> b |=> c"
assert P(S.TimedStatement(timing=S.CycleDelay(value=binary(a, "+", number())))) == "##(a + 1);"
print("ok")

## VLGPRT-02 · Every expression and part prints alone

In [ ]:
a, b, c = ref("a"), ref("b"), ref("c")
cases = [
    (S.MemberExpression(value=a, member=name("m")), "a.m"), (S.RangeSelect(value=a, left=number("7"), operator=":", right=number("0")), "a[7:0]"),
    (S.RangeSelect(value=a, left=number("0"), operator="+:", right=number("4")), "a[0 +: 4]"),
    (S.RealLiteral(spelling="1.5"), "1.5"), (S.TimeLiteral(spelling="1ns"), "1ns"), (S.UnbasedUnsizedLiteral(value="z"), "'z"),
    (S.StringLiteral(text="hi"), '"hi"'), (S.Concatenation(items=[a, number()]), "{a, 1}"),
    (S.Replication(count=number("2"), items=[a]), "{2{a}}"),
    (S.AssignmentPattern(items=[S.PatternItem(value=number()), S.PatternItem(key=a, value=number("2")), number("3")]), "'{default: 1, a: 2, 3}"),
    (S.AssignmentPattern(type=S.NamedType(name=name("t")), items=[number()]), "t'{1}"),
    (S.CallExpression(callee=a, arguments=[number(), S.NamedConnection(name=name("b"), value=number("2"))]), "a(1, .b(2))"),
    (S.SystemCall(name="$bits", arguments=[S.IntegerAtomType(keyword="int")]), "$bits(int)"), (S.SystemCall(name="$time"), "$time"),
    (S.CastExpression(type=S.ImplicitType(signing="unsigned"), value=a), "unsigned'(a)"),
    (S.MacroUsage(name=name("M"), arguments="1, 2"), "`M(1, 2)"), (S.MacroUsage(name=name("W")), "`W"), (S.DollarExpression(), "$"),
    (S.NameExpression(name=S.ScopedName(scope=name("p"), name=name("x"))), "p::x"),
    (S.RangeDimension(left=number("3"), right=number("0")), "[3:0]"), (S.UnsizedDimension(), "[]"),
    (S.AssociativeDimension(), "[*]"), (S.QueueDimension(bound=number("4")), "[$:4]"), (S.QueueDimension(), "[$]"),
    (S.IntegerVectorType(keyword="bit", signing="signed", dimensions=[S.SizeDimension(size=number("4"))]), "bit signed [4]"),
    (S.ImplicitType(dimensions=[S.RangeDimension(left=number("1"), right=number("0"))]), "[1:0]"),
    (S.EnumType(base=S.IntegerAtomType(keyword="int"), members=[S.EnumMember(name=name("A"), value=number())]), "enum int {A = 1}"),
    (S.StructType(keyword="union", members=[S.StructMember(type=S.KeywordType(keyword="string"), declarators=[S.VariableDeclarator(name=name("s"))])]), "union { string s; }"),
    (S.AnsiPort(name=name("p")), "p"), (S.InterfacePort(name=name("i")), "interface i"), (S.PortReference(name=name("r")), "r"),
    (S.IncludeDirective(path="types.svh", system=True), "`include <types.svh>"),
    (S.TimedStatement(timing=S.DelayControl(value=S.BinaryExpression(left=a, operator="+", right=number()))), "#(a + 1);"),
    (S.VirtualInterfaceType(interface=name("bus"), parameters=[number("4")], modport=name("mp")), "virtual bus #(4).mp"),
    (S.NewExpression(scope=S.ScopedName(scope=name("p"), name=S.ParameterizedName(name=name("c"), parameters=[number()])),
                     arguments=[a]), "p::c #(1)::new(a)"),
    (S.NewCopyExpression(value=S.CallExpression(callee=a)), "new a()"),
    (S.NewCopyExpression(value=S.BinaryExpression(left=a, operator="+", right=number())), "new (a + 1)"),
    (S.ForwardTypedefDeclaration(name=name("t")), "typedef t;"),
    (S.ConstraintPrototype(static=True, name=name("k")), "static constraint k;"),
    (S.NameExpression(name=S.LocalName(name=name("x"))), "local::x"),
    (S.RandomizeWithExpression(call=S.CallExpression(callee=ref("randomize"))), "randomize() with {}"),
    (S.DistExpression(value=a, items=[S.DistItem(value=number())]), "a dist {1}"),
    (S.DefaultSkew(input=S.ClockingSkew(edge="posedge")), "default input posedge;"),
    (S.DefaultSkew(output=S.ClockingSkew(delay=S.DelayControl(value=number("2")))), "default output #2;"),
    (S.BinsSelection(keyword="bins", name=name("s"), select=S.BinaryBinsSelect(left=a, operator="&&", right=S.BinaryBinsSelect(
        left=b, operator="||", right=c))), "bins s = a && (b || c);"),
    (S.BinsSelection(keyword="bins", name=name("s"), select=S.NotBinsSelect(operand=S.BinaryBinsSelect(
        left=a, operator="&&", right=b))), "bins s = !(a && b);"),
    (S.BinsSelection(keyword="bins", name=name("s"), select=S.FilteredBinsSelect(select=S.BinaryBinsSelect(
        left=a, operator="&&", right=b), filter=c)), "bins s = (a && b) with (c);"),
    (S.BinsSelection(keyword="bins", name=name("s"), select=S.FilteredBinsSelect(select=S.NotBinsSelect(operand=a),
                                                                               filter=c)), "bins s = !a with (c);"),
    (S.SystemCall(name="$time", attributes=[S.AttributeInstance(specs=[S.AttributeSpec(name=name("now"))])]),
     "$time (* now *) ()"),
]
for node, text in cases:
    assert P(node) == text, (P(node), text)
print("ok")

## VLGPRT-03 · Layout: blocks open on their header's line, `end else`, one item per line

In [ ]:
roundtrip("""module m;
    always_ff @(posedge clk) begin
        if (a) begin
            b <= 1;
        end else if (c) begin
            b <= 2;
        end else begin
            b <= 3;
        end
        if (a)
            b <= 4;
        else if (c)
            b <= 5;
        else
            b <= 6;
        unique case (a) inside
            1, [2:3]: b <= 1;
            4: begin
                b <= 2;
            end
            default: ;
        endcase
        for (int i = 0; i < 2; i++)
            a = i;
        do begin
        end while (a);
        do
            a = 1;
        while (a);
        wait (a);
        wait (a) b = 1;
        wait (a) begin
        end
        #1 a = 1;
        @(posedge clk) begin
        end
        assert (a);
        assert (a) b = 1; else b = 2;
        assert (a) begin
        end else begin
        end
        assert (a) else $error("x");
    end
endmodule
""")
roundtrip("""module m;
    if (W) begin : a
    end else if (V) begin : b
    end else
        assign x = 1;

    if (W)
        assign x = 2;
    else
        assign x = 3;

    case (W)
        1:
            assign x = 4;
        default: begin
        end
    endcase

    for (genvar i = 0; i < 2; i++) begin : g
    end : g

    for (j = 0; j < 2; j = j + 1)
        assign x = 5;
endmodule
""")
print("ok")

## VLGPRT-04 · Layout: design units, declarations and directives, with blank lines around what spans lines

In [ ]:
roundtrip("""// lead
`timescale 1ns / 1ps

/* the package */
package p;
    typedef struct packed {
        logic a;
        logic b;
    } s;
    var struct packed {
        logic a;
        logic b;
    } v;
    typedef union { int i; } u;

    function automatic int f(input int a = 1, output b);
        return a;
    endfunction : f

    task t;
        input a;
    endtask
endpackage : p

module m (a, b);
    input wire a;
    output reg b;

`ifdef X
    wire x;
`elsif Y
    wire y;
`else
    wire z;
`endif
endmodule

interface i
    import p::*;
#(
    parameter int W = 1,
    parameter type T = logic
) (
    input logic clk
);
    modport mp (input clk);
endinterface

program q #(
    localparam N = 1
);
endprogram
""")
text = P(SystemVerilog2023.parse("// lead\n`timescale 1ns / 1ps\n/* the package */\npackage p; typedef struct packed { logic a; logic b; } s; endpackage\n"))
assert text == "// lead\n`timescale 1ns / 1ps\n\n/* the package */\npackage p;\n    typedef struct packed {\n        logic a;\n        logic b;\n    } s;\nendpackage\n", text
print("ok")

## VLGPRT-05 · Printing validates the tree and checks the standard

In [ ]:
with raises(Errors.PrintError, match="a BinaryExpression needs a left"):
    P(S.BinaryExpression(operator="+", right=number()))
with raises(Errors.PrintError, match="PackageDeclaration is not Verilog, but this is Verilog-2005"):
    Verilog2005.print(S.PackageDeclaration(name=name("p")))
assert Verilog2005.print(S.ModuleDeclaration(keyword="module", name=name("m"))) == "module m;\nendmodule"
assert P(S.Comment(text=" c")) == "// c" and P(S.Comment(block=True, text=" c ")) == "/* c */"
print("ok")

## VLGPRT-06 · Layout: trailing comments, conditionals among statements, bodies that span lines

In [ ]:
roundtrip("""module m;  // trailing
    wire a;  /* after a */

    initial begin
        x = (a inside {1, 2}) && b;

`ifdef A
        x = 1;
`else
        x = 2;
`endif

        case (a)
            1:
                if (b)
                    x = 1;
                else
                    x = 2;
            default: ;
        endcase
        assert (a) $display("pass");
        assert (a) begin
            x = 1;
        end else $error("fail");
        ;
    end
endmodule
""")
print("ok")


## VLGPRT-07 · Layout: every option of headers, ports, declarations and statements

In [ ]:
roundtrip("""`define EMPTY

module a;
endmodule  // after a

// before b
module b
    import p::*;
(x, y);
    input var logic x;
    output y;
endmodule

module c
    import p::*;
(
    output var logic z = 1'b0,
    w
);
    wire #3 n;
    wire #(d + 1) m;
    const var static int k = 1;
    var v;
    static struct {
        logic a;
        logic b;
    } s;
    struct {
        logic a;
        logic b;
    } u;
    const struct packed {
        logic a;
        logic b;
    } [1:0] t = '0;
    typedef struct packed { logic a; } one;
    typedef enum {A} e;

    /* two
       lines */
    function int f(input var int i, j);
        return 1;
    endfunction

    task r;
        return;
    endtask

    initial begin
        ->> ev;
        disable named;
        disable fork;
        unique if (a)
            b = 1;
        for (; ; );
        for (int k = 0, m = 1; ; );
        #(a + 1) b = 1;
    end
endmodule
""")
print("ok")


## VLGPRT-08 · Layout: classes, their parameters, bases, members and prototypes

In [ ]:
roundtrip("""typedef class driver;

virtual class base #(
    type T = int,
    int N = 2
) extends parent #(T) implements i1, i2;
    local const static int count = 0;
    extern virtual function void f(int x);
    pure virtual task t;

    function new(int x = 0);
        super.new(x);
    endfunction
endclass : base

class derived extends base #(byte, 3)(1, .x(2));
endclass

interface class i1 extends j, k;
    pure virtual function void h;
endclass

module m #(
    int W = 8,
    type U = logic
);
    wire struct packed {
        logic a;
        logic b;
    } w;
endmodule
""")
print("ok")


## VLGPRT-09 · Layout: constraints, inline constraints and `randcase`

In [ ]:
roundtrip("""class c;
    rand int q[];

    constraint k {  // first
        x > 0;
        soft y < 10;
        x -> {
            y == 1;
        }
        x ->
            y == 2;
        if (x)
            y > 2;
        else if (z) {
            y < 3;
        } else
            y == 4;
        foreach (q[i]) {
            q[i] < 5;
        }
        solve x before y, q;
        disable soft y;
        unique {x, [1:2]};
        x dist {1 := 2, [3:4] :/ 5, 7, default :/ 1};

`ifdef SIM
        x < 9;
`endif
    }

    extern static constraint e;

    function void f;
        ok = randomize() with { // inline
x -> y; if (a) { b; } else c; if (d) e; foreach (q[i]) q[i] > 0; solve a before b; disable soft x; unique {a, b}; };
        ok = obj.randomize(x) with (x) {};
        ok = randomize() with {
`ifdef SIM
x > 1;
`endif
y < 3; };
        r = q.find(item) with (item > 1);
        randcase
            1: a = 1;
            2: begin
            end
        endcase
        void'(obj.randomize());
    endfunction
endclass

constraint c::e {
    x != y;
}
""")
print("ok")


## VLGPRT-10 · Layout: clocking blocks, property and sequence declarations, assertions and `let`

In [ ]:
roundtrip("""module m;
    default clocking cb @(posedge clk);
        default input #1step output #0;
        input #1 a, b;
        output negedge c;
        inout d = top.x;
        input #2 output #3 e;
    endclocking : cb

    global clocking gc @(posedge fast);
    endclocking

    default clocking @(negedge clk);
        property inner;
            if (a) b;
        endproperty
    endclocking

    default clocking cb;
    default disable iff (rst);

    sequence s1(x, int y = 1, local input logic z);
        int v;
        @(posedge clk) (a, v = x) ##1 b[*2] ##[1:3] c[->1] ##[0:$] d[=2:3] ##[1:$] e[*0:$] ##1 f[*1:$] ##(y) g;
    endsequence : s1

    sequence s2;
        ##1 a ##2 (b ##1 c)[*3] and first_match(d ##1 e, v++) or f intersect g within h throughout i;
    endsequence

    property p1(sequence q, untyped u);
        disable iff (rst) q |-> not (a until b) and s_eventually [1:$] c;
    endproperty

    property p2;
        if (a) strong(b ##1 c) else weak(d) implies nexttime [2] e iff always [1:3] f;
    endproperty

    property p3;
        accept_on (x) a s_until b or sync_reject_on (y) c until_with d;
    endproperty

    property p4;
        case (m) 0, 1: a |=> b; default: (c #=# d); endcase;
    endproperty

    a1: assert property (@(posedge clk) disable iff (rst) a |=> b) else $error("x");
    cover sequence (s1(1));
    restrict property (a ##1 @(negedge clk) b);
    assume property (s_nexttime a or @(negedge clk) b |-> c);
    assert #0 (x);
    let max(a, b = 1) = a > b ? a : b;

    initial begin
        ##2;
        ##1 cb.c <= 1;
        cb.c <= ##2 a;
        chk: assert (x);
        a2: assert property (p1(s1, 1));
        expect (@(posedge clk) a ##1 b) else $error;
        lab: x = 1;
    end
endmodule
""")
print("ok")


## VLGPRT-11 · Layout: covergroups, coverpoints with their bins, and crosses

In [ ]:
roundtrip("""class c;
    covergroup cg(int lo, ref int r) @(posedge clk);
        option.per_instance = 1;
        type_option.weight = 2;

        cp_a: coverpoint a iff (en) {
            bins low[4] = {[0:3], 5} iff (x);
            bins mid = {[4:$]} with (item % 2 == 0);
            wildcard bins w = {4'b1??0};
            illegal_bins bad = default;
            ignore_bins ig[] = (1 => 2 => 3), (4 => 5[*2:3] => 6[->1] => 7[=2]);
            bins tr = (1, 2 => 3);
            bins d = default sequence;
            option.at_least = 2;
            bins s = cp_b with (item > 1);
            bins e = a + 1;
        }

        coverpoint b;
        cp_c: coverpoint a + b;
        bit [3:0] cp_t: coverpoint a;

        x_ab: cross cp_a, b iff (en) {
            bins both = binsof(cp_a.low) && binsof(b) intersect {[1:2]};
            ignore_bins no = !binsof(cp_a) || binsof(cp_a.mid);
            bins w = x_ab with (cp_a > 1);
            bins p = (binsof(b));

            function int f;
            endfunction
        }

        cross cp_a, cp_c;
    endgroup : cg

    covergroup cg2 with function sample(int v);
        coverpoint v;
    endgroup

    covergroup cg3;
    endgroup
endclass
""")
print("ok")


## VLGPRT-12 · Layout: attributes start their item's or statement's line, and sit after their operator

In [ ]:
roundtrip("""(* top, version = "1.0" *) module m (
    (* keep *) input a,
    output b
);
    (* keep *) (* dont_touch = 1 *) wire w;

    (* full_case, parallel_case *) initial
        (* lab = 2 *) case (a)
            default: ;
        endcase

    assign b = a + (* mark *) w ? (* x *) a : b;

    initial begin
        x = f (* inline *) (1);
        y = $clog2 (* fast *) (8);
        z = - (* neg *) a;
        i (* inc *)++;
        ++ (* pre *) i;
        lab: (* attr *) x = 1;
    end

    class c;
        (* keep *) int p;
    endclass
endmodule
""")
print("ok")


## VLGPRT-13 · Layout: delays, streaming concatenations, repeated patterns, tolerance ranges, empty arguments, `$root`, `$unit` and `type()`

In [ ]:
roundtrip("""module m;
    wire #(1, 2, 3) w;
    assign #(1:2:3, 4) w = x;
    wire #(5) v;

    initial begin
        x = {>>{a, b}};
        x = {<< 8 {a with [0 +: 2]}};
        x = {<< byte {a with [1]}};
        x = {>> 4 {b with [1:2]}};
        x = '{2{a, b}};
        x = (1:2:3);
        x = a inside {[1 +/- 2], [3 +%- 4], [5:6]};
        x = f(a, , b);
        $display(a, , b);
        x = $root.top.a;
        x = {};
        x = $unit::a;
        y = type(a)'(b);
        #(1:2:3) x = 1;
    end

    var type(a) v2;
    var type(int) v3;
    u i (a, , b);
endmodule
""")
def typical():
    return S.MinTypMaxExpression(min=ref("a"), typ=ref("b"), max=ref("c"))


assign = S.ContinuousAssign(delay=S.DelayControl(value=typical(), fall=number("2")), assignments=[
    S.AssignmentExpression(target=ref("x"), operator="=", value=binary(typical(), "+", ref("y")))])
assert P(S.SourceText(items=[assign])) == "assign #(a:b:c, 2) x = (a:b:c) + y;\n"
print("ok")


## VLGPRT-14 · Layout: strengths, net types, aliases, `defparam`, time units, enumerations' ranges, randomized members, `const ref` and `ref static`, and type restrictions

In [ ]:
roundtrip("""package p;
    timeunit 1ns / 10ps;
    nettype logic [1:0] nt;
    nettype real total with p::sum;
    typedef enum {A[2] = 0, B[1:2], C} e;
    typedef struct { rand int a; } one;
    typedef struct {
        rand int a;
        randc bit [1:0] b;
    } two;

    function int f(const ref int a, ref static int b);
    endfunction
endpackage

module m #(
    parameter type enum T = e,
    type interface class I = i
);
    timeprecision 1ps;
    wire (strong0, weak1) w1;
    trireg (small) t;
    wire (highz1, pull0) vectored [1:0] w2;
    tri scalared [3:0] w3 = 1;
    assign (pull1, supply0) #1 a = b;
    alias a = b = c[1:0];
    defparam u.W = 2, v.X = 3;
    wire (supply0, supply1) struct packed {
        logic a;
        logic b;
    } pair;
endmodule
""")
print("ok")


## VLGPRT-15 · Layout: non-ANSI and explicit ports, and modports' subroutines, clocking blocks and explicit ports

In [ ]:
roundtrip("""module a (.p(x), .q(), .r({y, z}), , w[1:0], {u, v}, x[0], w[0 +: 1]);
    input [1:0] x, w;
    input y, z, u, v;
endmodule

module b (
    input .p(x + 1),
    .q(y),
    output logic z,
    .r()
);
endmodule

interface i;
    modport mp (import f, import g, export h, import task t(int a), import function int k, clocking cb, input .a(b), input c, output .d());
endinterface
""")
roundtrip("""interface i;
    modport mp ((* a *) input d, (* a *) input e, (* b = 1 *) output f, (* c *) import g, (* t *) clocking cb);
endinterface
""")
print("ok")

## VLGPRT-16 · Layout: specifiers, default arguments, empty items, interfaces' types, skipped `foreach` variables and `ifdef conditions

In [ ]:
roundtrip("""class :final c extends b(default);
    ;
    local typedef int t;
    protected typedef struct {
        int a;
        int b;
    } s;

    virtual function :initial :final void f;
    endfunction

    extern virtual function :extends void g;

    virtual task :final t2;
    endtask

    constraint :initial k {
        x > 0;
    }

    extern constraint :extends :final k2;

    constraint k3 {
        foreach (q[, j, , k])
            q[j] > 0;
    }
endclass

`ifdef (A && !B || (C -> D) || E <-> F)
module m; endmodule
`elsif (G)
module g; endmodule
`endif

module n (
    bus_if bus
);
    typedef bus.data_t t;

    initial
        foreach (q[i, ])
            x = 1;
endmodule
""")
print("ok")


## VLGPRT-17 · Layout: procedural continuous assignments, `wait fork`, `wait_order`, timed triggers, repeated event controls and elaboration tasks

In [ ]:
roundtrip("""module m;
    $error("bad %d", 1);
    $info;
    $fatal(1, , "x");

    generate
        if (1)
            $warning("w");
    endgenerate

    initial begin
        force a[0] = 1;
        release a;
        assign b = 1;
        deassign b;
        wait fork;
        wait_order (a, b.c) x = 1; else y = 1;
        wait_order (a, b);
        wait_order (a) else begin
            y = 1;
            z = 2;
        end
        ->> #1 e;
        ->> @(posedge clk) e;
        ->> e;
        a <= repeat (2) @(posedge clk) b;
    end
endmodule
""")
print("ok")


## VLGPRT-18 · Layout: DPI imports and exports, `bind`, `extern` design units and `.*` ports

In [ ]:
roundtrip("""module m;
    import "DPI-C" function int f(int a);
    import "DPI-C" context task t;
    import "DPI-C" pure c_name = function int g;
    import "DPI" function void h;
    export "DPI-C" function f;
    export "DPI-C" e = task t;
    bind n mon k ();
endmodule

bind m checker_unit c (.*);
bind m: u1, u2 mon #(1) i (.a(b)), j ();
bind top.m mon k ();

extern module e #(
    parameter W = 1
) (
    input a
);

extern interface ei (
    input a
);

extern program ep;

extern module en (a, b);

module e (.*);
endmodule
""")
roundtrip("""interface i;
    extern task probe(input int n);
    extern forkjoin task sweep;
    extern function int f;
endinterface

module m (
    i.mp bus
);
    task bus.probe(input int n);
    endtask
endmodule
""")
print("ok")

## VLGPRT-19 · Layout: a production on a line, but for code that spans lines

In [ ]:
roundtrip("""module m;
    initial
        randsequence (main)
            main : first second(2) | third := 2 | rand join (0.5) first third | rand join second(1) first := 1 { x = 3; };
            first : {
                x = 1;  // one
            };
            second(int n = 1) : if (a) third else first | repeat (2) third | case (b) 0, 1: first; default: second(3); endcase | if (c) first;
            third : {
                x = 2;
                if (y) begin
                    z = 1;
                end
            } first;
            void f : { return; };
            int g : { x = 1; return 1; };
        endsequence

    initial
        randsequence ()
            a : b;
            b : { };
        endsequence
endmodule
""")
print("ok")


## VLGPRT-20 · Layout: checkers, their instances, package exports and nets of user-defined types

In [ ]:
roundtrip("""package p;
    checker pc(a);
        assert property (a);
    endchecker

    export q::*;
    export q::x, r::y;
    export *::*;
endpackage

checker c(input logic clk, a, output bit b = 0, property pr, sequence s = a ##1 a, untyped u, event e = $inferred_clock);
    default clocking @(posedge clk);
    endclocking

    rand bit r;
    rand const int k;
    bit v = 0;

    checker inner;
    endchecker
endchecker : c

module m;
    nettype logic nt;
    nt #1 w, v = 1;
    nt w2;
    c c1 (.clk(clk), .a(x));
    p::pc c4 (x);

    always @(posedge clk) begin
        c c2 (clk, x), c3 (.*);
        p::pc c5 (x);
    end
endmodule
""")
print("ok")


## VLGPRT-21 · Layout: tagged unions and values, `case matches`, predicates and patterns; a tagged value is parenthesized where it is an operand

In [ ]:
roundtrip("""module m;
    typedef union tagged packed {
        int a;
        void b;
    } t;
    typedef union soft {
        int a;
        bit [31:0] c;
    } s;

    initial begin
        x = tagged a 5;
        x = tagged b;
        f(tagged a (1 + 2), tagged a tagged b 1);
        y = (tagged a 5) + 1;
        case (x) matches
            tagged a .v &&& v > 0: ;
            tagged b: ;
            .*: ;
            default: ;
        endcase
        if (x matches tagged a .v &&& v > 1);
        y = x matches tagged a .v &&& v > 1 ? v : 0;
        case (p) matches
            '{.q, 2}: ;
            '{f: .r, g: tagged a .*}: ;
        endcase
        if (a &&& b);
    end
endmodule
""")
tagged = S.TaggedExpression(member=name("a"), value=binary(ref("x"), "+", number()))
assert P(S.SourceText(items=[S.ContinuousAssign(assignments=[S.AssignmentExpression(
    target=ref("y"), operator="=", value=binary(tagged, "*", number("2")))])])) == "assign y = (tagged a (x + 1)) * 2;\n"
print("ok")


## VLGPRT-22 · Layout: user-defined primitives, their tables, and gate instances

In [ ]:
roundtrip("""primitive mux (output o, input s, a, b);
    table
        // s a b : o
        0 1 ? : 1;
        1 ? 0 : 0;
    endtable
endprimitive : mux

primitive latch (q, d, en);
    output q;
    reg q;
    input d, en;
    initial q = 1'b0;
    table
        (01) 1 : ? : 1;
        r ? : 0 : -;
        * 0 : ? : -;
        (0x) x : 1 : x;
    endtable
endprimitive

primitive dff (output reg q = 0, input d, clk);
    table
        0 r : ? : 0;
    endtable
endprimitive

module m;
    and #(1, 2) g1 (o, a, b), g2 (o2, a, b);
    nand (strong0, weak1) (o3, a, b);
    bufif0 #3 b1[1:0] (o4, a, en);
    pullup (p);
    pulldown (weak0) pd (q);
    tranif1 t1 (x, y, en);
    not (n1, n2, a);
    mux u1 (o, s, a, b);
    latch #1 (q, d, en);
endmodule
""")
roundtrip("""extern primitive p (output o, input a);

extern primitive q (o, a);

primitive p (.*);
    output o;
    input a;
    table
        0 : 1;
    endtable
endprimitive
""")
roundtrip("""primitive p (o, a);
    (* x *) output o;
    (* y = 1 *) input a;
    table
        0 : 1;
    endtable
endprimitive
""")
print("ok")

## VLGPRT-23 · Layout: a specify block, a path and a timing check each on a line

In [ ]:
roundtrip("""module m (
    input a,
    b,
    clk,
    d,
    en,
    output y,
    q
);
    specify
        // the delays of the paths
        specparam [7:0] w = 8, PATHPULSE$a$y = (1, 2);
        specparam tRise = 1:2:3, tFall = 2;
        (a => y) = (tRise, tFall);
        (a, b *> y, q) = 3;
        (a +=> y) = 1;
        (a -*> y) = 1;
        (b +*> y) = 1;
        if (b) (a => y) = 1;
        ifnone (a => y) = 2;
        (posedge clk => (q +: d)) = (1, 2);
        (negedge clk *> (q -: d)) = 2;
        (clk => (q : d)) = 2;
        if (en) (posedge clk => (q : d)) = 1;
        $setup(d, posedge clk, 2);
        $hold(posedge clk &&& en, d, 1, notifier);
        $setuphold(posedge clk, d, 1, 2, , , , dclk, dd);
        $period(edge [01, 10] clk, 10);
        $width(negedge clk, 5);
        $hold(posedge clk, d &&& en, 1);
        pulsestyle_ondetect y, q;
        noshowcancelled y;
    endspecify

    specparam s = 1;
endmodule
""")
print("ok")


## VLGPRT-24 · Layout: covergroup extensions, block events, `matches`, `super.new(default)`, interface ports, attributes everywhere and sequence arguments

In [ ]:
roundtrip("""class b;
    covergroup g;
        coverpoint x;
    endgroup
endclass

class d extends b;
    covergroup extends g;
        coverpoint y;
    endgroup

    function new;
        super.new(default);
    endfunction
endclass

module m (a);
    bus_if.mp a;
    typedef struct {
        (* keep *) int x;
        rand bit y;
    } t;

    function int f((* unused *) int x, input int y);
    endfunction

    u i1 ((* a *) .x(y), (* b *) z);

    property p(sequence s);
        s;
    endproperty

    sequence q(s);
        s;
    endsequence

    assert property (p(a ##1 b));
    assert property (q(a[*2]));
    assert property (p(.s(a ##1 b)));

    default clocking cb @(posedge clk);
        input x;

        (* prop *) property pc;
            x;
        endproperty
    endclocking

    covergroup g2 @@(begin f or end t.run);
        (* point *) coverpoint x {
            (* bin *) bins low = {0};
        }

        x: cross a, b {
            (* sel *) bins s = x matches 2;
            bins w = binsof(a) with (a > 1) matches 0;

            (* helper *) function int h;
                return 1;
            endfunction
        }
    endgroup
endmodule
""")
print("ok")


## VLGPRT-25 · Layout: configurations, a rule on a line

In [ ]:
roundtrip("""config cfg;
    localparam W = 8;
    design work.top lib2.other;
    // the libraries
    default liblist work lib2;
    instance top.u1 liblist lib3;
    instance top.u2 use lib4.cell2:config;
    cell lib1.adder use work.fast_adder;
    cell mux liblist lib2;
    instance top.u3 use #(.W(16));
    instance top.u4.u5 use fast;
    cell slow liblist;
endconfig : cfg

config plain;
    design top;
    default liblist work;
endconfig

module m (
    (* b *) .p(x),
    (* c *) input y
);
endmodule
""")
print("ok")
